In [9]:
import pandas as pd

dados ='Dados Brutos/COTAHIST_A2022.TXT'

df=pd.read_fwf(dados)

df.head

<bound method NDFrame.head of          00COTAHIST.2022BOVESPA             20221229 Unnamed: 2 Unnamed: 3  \
0             012022010302GNDI3    010INTERMEDICA ON        NaN         NM   
1             012022010302ABEV3    010AMBEV S/A   ON        NaN        NaN   
2            012022010302MODL11   010MODALMAIS   UNT        NaN         N2   
3             012022010302TASA4    010TAURUS ARMASPN        NaN         N2   
4             012022010302CRIV4    010ALFA FINANC PN        NaN        NaN   
...                         ...                  ...        ...        ...   
2117436      012022120662BRFS3T    030BRF SA      ON        NaN         NM   
2117437      012022121262BRFS3T    030BRF SA      ON        NaN         NM   
2117438      012022121962BRFS3T    030BRF SA      ON        NaN         NM   
2117439      012022122162BRFS3T    030BRF SA      ON        NaN         NM   
2117440  99COTAHIST.2022BOVESPA  2022122900002117442        NaN        NaN   

        Unnamed: 4               

In [ ]:
import pandas as pd

# Caminho do arquivo
arquivo = 'Dados Brutos/COTAHIST_A2022.TXT'

# Definição das colunas com posições (início, fim) - base 1
colunas = [
    ('tipo_registro', 1, 2),
    ('data', 3, 10),
    ('codigo_bdi', 11, 12),
    ('codigo', 13, 24),
    ('tipo_mercado', 25, 27),
    ('nome_empresa', 28, 39),
    ('especificacao', 40, 49),
    ('prazo', 50, 52),
    ('moeda', 53, 56),
    ('preco_abertura', 57, 69),
    ('preco_maximo', 70, 82),
    ('preco_minimo', 83, 95),
    ('preco_medio', 96, 108),
    ('preco_ultimo', 109, 121),
    ('melhor_oferta_compra', 122, 134),
    ('melhor_oferta_venda', 135, 147),
    ('numero_negocios', 148, 152),
    ('quantidade_titulos', 153, 170),
    ('volume_total', 171, 188),
    ('preco_exercicio', 189, 201),
    ('indicador_correcao', 202, 202),
    ('data_vencimento', 203, 210),
    ('fator_cotacao', 211, 217),
    ('preco_exercicio_pontos', 218, 230),
    ('isin', 231, 241),
    ('numero_distribuicao', 242, 244)
]

# Calcular as larguras a partir das posições
widths = [fim - inicio + 1 for _, inicio, fim in colunas]
nomes = [col for col, _, _ in colunas]

# Ler o arquivo com largura fixa
df = pd.read_fwf(
    arquivo,
    widths=widths,
    names=nomes,
    dtype=str,                # tudo como string para preservar zeros à esquerda
    encoding='latin1'         # ou 'utf-8', dependendo do encoding do arquivo
)

# Filtrar apenas registros de cotação histórica (tipo_registro = '01')
df = df[df['tipo_registro'] == '01'].copy()

# Converter datas
df['data'] = pd.to_datetime(df['data'], format='%d%m%Y', errors='coerce')

# Converter campos numéricos (preços e volumes têm 2 casas decimais implícitas)
colunas_preco = [
    'preco_abertura', 'preco_maximo', 'preco_minimo', 'preco_medio',
    'preco_ultimo', 'melhor_oferta_compra', 'melhor_oferta_venda',
    'preco_exercicio', 'volume_total'
]
for col in colunas_preco:
    df[col] = pd.to_numeric(df[col], errors='coerce') / 100

# Converter quantidades
df['numero_negocios'] = pd.to_numeric(df['numero_negocios'], errors='coerce')
df['quantidade_titulos'] = pd.to_numeric(df['quantidade_titulos'], errors='coerce')

# Opcional: filtrar apenas ações do lote padrão (código_bdi = '02')
# df = df[df['codigo_bdi'] == '02']

# Salvar como Excel
arquivo_saida = 'Dados/COTAHIST_A2022_ajustado.csv'
df.to_csv(arquivo_saida, index=False, encoding='utf-8-sig')
print(f"Arquivo CSV salvo em: {arquivo_saida}")

print(f"Arquivo salvo em: {arquivo_saida}")
print(f"Total de registros: {len(df)}")

Arquivo CSV salvo em: Dados/COTAHIST_A2022_ajustado.csv
Arquivo salvo em: Dados/COTAHIST_A2022_ajustado.csv
Total de registros: 2117440


In [8]:
df.head(10000).to_csv('amostra.csv', index=False)

In [10]:
import pandas as pd

arquivo = 'Dados Brutos/COTAHIST_A2022.TXT'

# Larguras oficiais do Layout da B3 (COTAHIST)
larguras = [2, 8, 2, 12, 3, 12, 10, 3, 4, 13, 13, 13, 13, 13, 13, 13, 5, 18, 18, 9, 7, 13, 8, 12]

nomes = [
    'tipo_registro', 'data', 'cod_bdi', 'codigo', 'tipo_mercado', 'nome_empresa', 
    'especificacao', 'prazo', 'moeda', 'preco_abertura', 'preco_maximo', 'preco_minimo', 
    'preco_medio', 'preco_ultimo', 'oferta_compra', 'oferta_venda', 'num_negocios', 
    'quantidade_titulos', 'volume_total', 'preco_exercicio', 'ind_correcao', 
    'data_vencimento', 'fator_cotacao', 'preco_pontos'
]

# Ler o arquivo
df = pd.read_fwf(arquivo, widths=larguras, names=nomes, skiprows=1, skipfooter=1, encoding='latin1')

# --- LIMPEZA DOS DADOS ---

# 1. Remover espaços em branco dos códigos e nomes
df['codigo'] = df['codigo'].str.strip()
df['nome_empresa'] = df['nome_empresa'].str.strip()

# 2. Converter Preços (Dividir por 100 para ter as casas decimais)
cols_preco = ['preco_abertura', 'preco_maximo', 'preco_minimo', 'preco_medio', 'preco_ultimo']
for col in cols_preco:
    df[col] = df[col] / 100

# 3. Converter Data
df['data'] = pd.to_datetime(df['data'], format='%Y%m%d')

# --- O PULO DO GATO: FILTRAR O QUE VOCÊ PRECISA ---

# Escolha o contrato (Ex: Milho Maio 2022)
meu_contrato = 'CCMK22'
df_hedge = df[df['codigo'] == meu_contrato].copy()

# 4. Salvar apenas o que interessa (Arquivo leve que abre no Excel)
df_hedge.to_csv('Dados/Hedge_Milho_Maio.csv', index=False, encoding='utf-8-sig')

print(f"Sucesso! Foram encontradas {len(df_hedge)} linhas para o contrato {meu_contrato}.")

Sucesso! Foram encontradas 0 linhas para o contrato CCMK22.


In [19]:
import pandas as pd

arquivo = 'Dados Brutos/COTAHIST_A2022.TXT'

dados = []

with open(arquivo, 'r', encoding='latin1') as f:
    for linha in f:
        
        linha = linha.rstrip('\n').rstrip('\r')
        
        if not linha.startswith('01'):
            continue
        
        registro = {
            'tipo_registro': linha[0:2],
            'data': linha[2:10],
            'codigo_bdi': linha[10:12],
            'codigo': linha[12:24].strip(),
            'tipo_mercado': linha[24:27],
            'nome_empresa': linha[27:39].strip(),
            'especificacao': linha[39:49].strip(),
            'prazo': linha[49:52].strip(),
            'moeda': linha[52:56].strip(),
            
            'preco_abertura': linha[56:69],
            'preco_maximo': linha[69:82],
            'preco_minimo': linha[82:95],
            'preco_medio': linha[95:108],
            'preco_ultimo': linha[108:121],
            'melhor_oferta_compra': linha[121:134],
            'melhor_oferta_venda': linha[134:147],
            
            'numero_negocios': linha[147:152],
            'quantidade_titulos': linha[152:170],
            'volume_total': linha[170:188],
            
            'preco_exercicio': linha[188:201],
            'indicador_correcao': linha[201:202],
            'data_vencimento': linha[202:210],
            'fator_cotacao': linha[210:217],
            'preco_exercicio_pontos': linha[217:230],
            'isin': linha[230:242].strip(),
            'numero_distribuicao': linha[242:245]
        }
        
        dados.append(registro)

df = pd.DataFrame(dados)

print(f'Total de registros: {len(df)}')
df.head()

Total de registros: 2117440


,tipo_registro,data,codigo_bdi,codigo,tipo_mercado,nome_empresa,especificacao,prazo,moeda,preco_abertura,...,numero_negocios,quantidade_titulos,volume_total,preco_exercicio,indicador_correcao,data_vencimento,fator_cotacao,preco_exercicio_pontos,isin,numero_distribuicao
0,01,20220103,02,GNDI3,010,INTERMEDICA,ON NM,,R$,0000000005970,...,20535,000000000003960200,000000023351263100,0000000000000,0,99991231,0000001,0000000000000,BRGNDIACNOR2,103
1,01,20220103,02,ABEV3,010,AMBEV S/A,ON,,R$,0000000001542,...,43784,000000000023833600,000000036454098800,0000000000000,0,99991231,0000001,0000000000000,BRABEVACNOR1,125
2,01,20220103,02,MODL11,010,MODALMAIS,UNT N2,,R$,0000000001080,...,03729,000000000000725000,000000000803365000,0000000000000,0,99991231,0000001,0000000000000,BRMODLCDAM13,103
3,01,20220103,02,TASA4,010,TAURUS ARMAS,PN N2,,R$,0000000002500,...,02665,000000000000610900,000000001506876600,0000000000000,0,99991231,0000001,0000000000000,BRTASAACNPR4,100
4,01,20220103,02,CRIV4,010,ALFA FINANC,PN,,R$,0000000000568,...,00010,000000000000005800,000000000003296500,0000000000000,0,99991231,0000001,0000000000000,BRCRIVACNPR1,206


In [14]:
import pandas as pd
import os

# =========================
# CONFIGURAÇÃO
# =========================

pasta = 'Dados Brutos'

codigos_busca = [
    'ccmf2004',
    'ccmk2022',
    'sjch2006',
    'icfu1996',
    'ethg2010',
    'bgif2007'
]

# =========================
# PADRONIZAR CÓDIGOS
# =========================

def padronizar_codigo(cod):
    cod = cod.upper()
    base = cod[:3]
    mes = cod[3]
    ano = cod[-2:]
    return f"{base}{mes}{ano}"

codigos_formatados = [padronizar_codigo(c) for c in codigos_busca]

print("Códigos formatados:", codigos_formatados)

# =========================
# FUNÇÃO DE LEITURA
# =========================

def ler_cotahist(caminho_arquivo):
    dados = []
    
    with open(caminho_arquivo, 'r', encoding='latin1') as f:
        for linha in f:
            
            linha = linha.rstrip('\n').rstrip('\r')
            
            if not linha.startswith('01'):
                continue
            
            registro = {
                'tipo_registro': linha[0:2],
                'data': linha[2:10],
                'codigo_bdi': linha[10:12],
                'codigo': linha[12:24].strip(),
                'tipo_mercado': linha[24:27],
                'nome_empresa': linha[27:39].strip(),
                'especificacao': linha[39:49].strip(),
                'prazo': linha[49:52].strip(),
                'moeda': linha[52:56].strip(),
                
                'preco_abertura': linha[56:69],
                'preco_maximo': linha[69:82],
                'preco_minimo': linha[82:95],
                'preco_medio': linha[95:108],
                'preco_ultimo': linha[108:121],
                'melhor_oferta_compra': linha[121:134],
                'melhor_oferta_venda': linha[134:147],
                
                'numero_negocios': linha[147:152],
                'quantidade_titulos': linha[152:170],
                'volume_total': linha[170:188],
                
                'preco_exercicio': linha[188:201],
                'indicador_correcao': linha[201:202],
                'data_vencimento': linha[202:210],
                'fator_cotacao': linha[210:217],
                'preco_exercicio_pontos': linha[217:230],
                'isin': linha[230:242].strip(),
                'numero_distribuicao': linha[242:245]
            }
            
            dados.append(registro)
    
    df = pd.DataFrame(dados)
    
    # =========================
    # TRATAMENTO
    # =========================
    
    df['data'] = pd.to_datetime(df['data'], format='%Y%m%d', errors='coerce')
    df['data_vencimento'] = pd.to_datetime(df['data_vencimento'], format='%Y%m%d', errors='coerce')
    
    colunas_preco = [
        'preco_abertura', 'preco_maximo', 'preco_minimo',
        'preco_medio', 'preco_ultimo',
        'melhor_oferta_compra', 'melhor_oferta_venda',
        'preco_exercicio', 'volume_total'
    ]
    
    for col in colunas_preco:
        df[col] = pd.to_numeric(df[col], errors='coerce') / 100
    
    df['numero_negocios'] = pd.to_numeric(df['numero_negocios'], errors='coerce')
    df['quantidade_titulos'] = pd.to_numeric(df['quantidade_titulos'], errors='coerce')
    df['fator_cotacao'] = pd.to_numeric(df['fator_cotacao'], errors='coerce')
    
    return df

# =========================
# LER TODOS OS ARQUIVOS
# =========================

dfs = []

arquivos = os.listdir(pasta)

for arquivo in arquivos:
    
    caminho = os.path.join(pasta, arquivo)
    
    print(f'Lendo: {caminho}')
    
    try:
        df_temp = ler_cotahist(caminho)
        
        # Extrair ano do nome do arquivo
        ano = ''.join(filter(str.isdigit, arquivo))
        df_temp['ano'] = int(ano) if ano else None
        
        dfs.append(df_temp)
    
    except Exception as e:
        print(f'Erro ao ler {arquivo}: {e}')

# =========================
# UNIFICAR BASE
# =========================

df_final = pd.concat(dfs, ignore_index=True)

# Limpeza
df_final['codigo'] = df_final['codigo'].str.strip()
df_final = df_final.drop_duplicates()

print(f'\nTotal de registros geral: {len(df_final)}')

# =========================
# FILTRO EXATO
# =========================

df_exato = df_final[df_final['codigo'].isin(codigos_formatados)]

print(f'Total encontrados (exato): {len(df_exato)}')

# =========================
# FILTRO FLEXÍVEL
# =========================

prefixos = ['CCM', 'BGI', 'ICF', 'SJC', 'ETH']

df_flex = df_final[df_final['codigo'].str[:3].isin(prefixos)]

print(f'Total encontrados (flexível): {len(df_flex)}')

# =========================
# SALVAR
# =========================

df_final.to_parquet('cotacoes_todos_anos.parquet')
df_exato.to_parquet('commodities_exatas.parquet')
df_flex.to_parquet('commodities_flexivel.parquet')

# =========================
# VISUALIZAÇÃO
# =========================

print("\nExemplo (exato):")
print(df_exato[['codigo', 'data', 'preco_ultimo']].head())

print("\nExemplo (flexível):")
print(df_flex[['codigo', 'data', 'preco_ultimo']].head())

Códigos formatados: ['CCMF04', 'CCMK22', 'SJCH06', 'ICFU96', 'ETHG10', 'BGIF07']
Lendo: Dados Brutos\COTAHIST.A1996
Lendo: Dados Brutos\COTAHIST_A2004.TXT
Lendo: Dados Brutos\COTAHIST_A2006.TXT
Lendo: Dados Brutos\COTAHIST_A2007.TXT
Lendo: Dados Brutos\COTAHIST_A2010.TXT
Lendo: Dados Brutos\COTAHIST_A2022.TXT

Total de registros geral: 3161164
Total encontrados (exato): 0
Total encontrados (flexível): 2016

Exemplo (exato):
Empty DataFrame
Columns: [codigo, data, preco_ultimo]
Index: []

Exemplo (flexível):
      codigo       data  preco_ultimo
215    ICF 4 1996-01-02         21.00
9327   ICF 4 1996-02-02         15.05
18267  ICF 4 1996-03-07        110.01
18647  ICF 4 1996-03-08         75.00
18899  ICF 4 1996-03-08         80.00


In [18]:
import pandas as pd

df = pd.read_parquet('commodities_flexivel.parquet')

df.head()

,tipo_registro,data,codigo_bdi,codigo,tipo_mercado,nome_empresa,especificacao,prazo,moeda,preco_abertura,...,quantidade_titulos,volume_total,preco_exercicio,indicador_correcao,data_vencimento,fator_cotacao,preco_exercicio_pontos,isin,numero_distribuicao,ano
215,01,1996-01-02,96,ICF 4,020,C FABRINI,PN *,,R$,21.00,...,500,10.50,0.0,0,NaT,1000,0000000000000,BRICFBACNPR8,119,1996
9327,01,1996-02-02,96,ICF 4,020,C FABRINI,PN *,,R$,20.00,...,800,15.35,0.0,0,NaT,1000,0000000000000,BRICFBACNPR8,119,1996
18267,01,1996-03-07,02,ICF 4,010,C FABRINI,PN *,,R$,110.01,...,157000,17271.57,0.0,0,NaT,1000,0000000000000,BRICFBACNPR8,119,1996
18647,01,1996-03-08,02,ICF 4,010,C FABRINI,PN *,,R$,110.00,...,176000,18940.00,0.0,0,NaT,1000,0000000000000,BRICFBACNPR8,119,1996
18899,01,1996-03-08,96,ICF 4,020,C FABRINI,PN *,,R$,80.00,...,262,20.96,0.0,0,NaT,1000,0000000000000,BRICFBACNPR8,119,1996


In [21]:
import pandas as pd

# Caminho do arquivo original
arquivo = 'Dados Brutos/COTAHIST_A2022.TXT'

# 1. Definição correta e sincronizada (26 colunas)
colunas_config = [
    ('tipo_registro', 2),
    ('data', 8),
    ('codigo_bdi', 2),
    ('codigo', 12),
    ('tipo_mercado', 3),
    ('nome_empresa', 12),
    ('especificacao', 10),
    ('prazo', 3),
    ('moeda', 4),
    ('preco_abertura', 13),
    ('preco_maximo', 13),
    ('preco_minimo', 13),
    ('preco_medio', 13),
    ('preco_ultimo', 13),
    ('melhor_oferta_compra', 13),
    ('melhor_oferta_venda', 13),
    ('numero_negocios', 5),
    ('quantidade_titulos', 18),
    ('volume_total', 18),
    ('preco_exercicio', 13),
    ('indicador_correcao', 1),
    ('data_vencimento', 8),
    ('fator_cotacao', 7),
    ('preco_exercicio_pontos', 13),
    ('isin', 12),
    ('numero_distribuicao', 3)
]

# Extrair larguras e nomes garantindo que tenham o mesmo tamanho
widths = [c[1] for c in colunas_config]
nomes = [c[0] for c in colunas_config]

# 2. Ler o arquivo (dtype=str para evitar perda de zeros)
# skiprows=1 e skipfooter=1 para ignorar cabeçalho e rodapé da B3
df = pd.read_fwf(
    arquivo, 
    widths=widths, 
    names=nomes, 
    dtype=str, 
    encoding='latin1',
    skiprows=1,
    skipfooter=1
)

# --- 3. LIMPEZA E TRATAMENTO ---

# Limpar espaços nos códigos (essencial para o filtro funcionar)
df['codigo'] = df['codigo'].str.strip()

# Converter Data (B3 usa AAAAMMDD)
df['data'] = pd.to_datetime(df['data'], format='%Y%m%d', errors='coerce')

# Converter Preços (Dividir por 100 para centavos)
colunas_preco = [
    'preco_abertura', 'preco_maximo', 'preco_minimo', 'preco_medio', 
    'preco_ultimo', 'volume_total'
]
for col in colunas_preco:
    df[col] = pd.to_numeric(df[col], errors='coerce') / 100

# --- 4. FILTRAR O QUE VOCÊ PRECISA (CCM = Milho | ICF = Café) ---
# Aqui você cria sua "Tabela de Busca" para o FUT ADV
df_commodities = df[df['codigo'].str.contains('CCM|ICF', na=False)].copy()

# Salvar apenas o que interessa
df_commodities.to_csv('Tabela_Busca_FUT_ADV.csv', index=False, encoding='utf-8-sig')

print(f"Sucesso! Tabela criada com {len(df_commodities)} registros de Milho/Café.")
print(df_commodities[['data', 'codigo', 'preco_ultimo']].head())

Sucesso! Tabela criada com 25 registros de Milho/Café.
              data  codigo  preco_ultimo
1036127 2022-11-30  CCME11        102.14
1040423 2022-12-05  CCME11        102.14
1042506 2022-11-25  CCME11        102.14
1047928 2022-12-07  CCME11        102.14
1059501 2022-12-08  CCME11        102.00


In [22]:
import pandas as pd

# ==================================================
# Definição das colunas (igual à sua lista)
# ==================================================
colunas = [
    ('tipo_registro', 1, 2),
    ('data', 3, 10),
    ('codigo_bdi', 11, 12),
    ('codigo', 13, 24),
    ('tipo_mercado', 25, 27),
    ('nome_empresa', 28, 39),
    ('especificacao', 40, 49),
    ('prazo', 50, 52),
    ('moeda', 53, 56),
    ('preco_abertura', 57, 69),
    ('preco_maximo', 70, 82),
    ('preco_minimo', 83, 95),
    ('preco_medio', 96, 108),
    ('preco_ultimo', 109, 121),
    ('melhor_oferta_compra', 122, 134),
    ('melhor_oferta_venda', 135, 147),
    ('numero_negocios', 148, 152),
    ('quantidade_titulos', 153, 170),
    ('volume_total', 171, 188),
    ('preco_exercicio', 189, 201),
    ('indicador_correcao', 202, 202),
    ('data_vencimento', 203, 210),
    ('fator_cotacao', 211, 217),
    ('preco_exercicio_pontos', 218, 230),
    ('isin', 231, 241),
    ('numero_distribuicao', 242, 244)
]

# Calcular larguras
widths = [fim - inicio + 1 for _, inicio, fim in colunas]
nomes = [c[0] for c in colunas]

# ==================================================
# Leitura do arquivo
# ==================================================
arquivo = 'Dados Brutos/COTAHIST_A2004.TXT'

df = pd.read_fwf(
    arquivo,
    widths=widths,
    names=nomes,
    dtype=str,               # tudo como string inicialmente
    encoding='latin1'        # ou 'utf-8' conforme o arquivo
)

# Filtrar apenas registros de cotação histórica (tipo_registro = '01')
df = df[df['tipo_registro'] == '01'].copy()

# Converter datas
df['data'] = pd.to_datetime(df['data'], format='%d%m%Y', errors='coerce')

# Converter campos numéricos (preços e volumes têm 2 casas decimais implícitas)
colunas_preco = [
    'preco_abertura', 'preco_maximo', 'preco_minimo', 'preco_medio',
    'preco_ultimo', 'melhor_oferta_compra', 'melhor_oferta_venda',
    'preco_exercicio', 'volume_total'
]
for col in colunas_preco:
    df[col] = pd.to_numeric(df[col], errors='coerce') / 100

# Converter quantidades
df['numero_negocios'] = pd.to_numeric(df['numero_negocios'], errors='coerce')
df['quantidade_titulos'] = pd.to_numeric(df['quantidade_titulos'], errors='coerce')

# ==================================================
# Salvar como CSV (mais leve que Excel)
# ==================================================
arquivo_saida = 'Dados/COTAHIST_A2004_ajustado.csv'
df.to_csv(arquivo_saida, index=False, encoding='utf-8-sig')
print(f"Arquivo salvo: {arquivo_saida}")
print(f"Total de linhas: {len(df)}")

Arquivo salvo: Dados/COTAHIST_A2004_ajustado.csv
Total de linhas: 166937
